# CookMatch — Colab Full Run

Clone repo → Kaggle auth → download Food.com → train pipeline → sample recommendations + ablation.

**Runtime:** CPU is enough. Full 231k recipe filter can take 30–90 min on free Colab.

In [ ]:
# 1) Clone repo
REPO_URL = "https://github.com/YUV3571/cookmatch-recipe-recommender.git"
REPO_DIR = "/content/cookmatch-recipe-recommender"

import os
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    %cd $REPO_DIR
    !git pull origin main

%cd $REPO_DIR
print("Working dir:", os.getcwd())

In [ ]:
# 2) Install dependencies
!pip install -q kagglehub pandas numpy scipy pyarrow

In [ ]:
# 3) Kaggle authentication
import os

# Option A (recommended): Kaggle API token
# Kaggle → Settings → API → Create New Token → copy KGAT_... value
os.environ["KAGGLE_API_TOKEN"] = "KGAT_your_token_here"  # paste your token

# Option B: upload kaggle.json instead (comment out Option A first)
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

assert os.environ.get("KAGGLE_API_TOKEN", "").startswith("KGAT_"), "Set a valid KAGGLE_API_TOKEN"
print("Kaggle token configured.")

In [ ]:
# 4) Download dataset
import os
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# sanity check — src/data must exist in cloned repo
assert os.path.isdir(os.path.join(REPO_DIR, "src", "data")), (
    "Missing src/data in repo. Run: !cd {REPO_DIR} && git pull origin main"
)

import kagglehub
from src.data.loader import get_dataset_path, load_interaction_split, load_recipes

dataset_path = get_dataset_path()
print("Dataset path:", dataset_path)

recipes = load_recipes(nrows=5000, columns=["id", "name", "ingredients", "minutes", "tags"])
train = load_interaction_split("train")
print("recipes sample:", len(recipes))
print("train interactions:", len(train))

In [ ]:
# 5) Train Stage 3 recommender + demo recommendations
from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext
from src.recommend.stage3 import Stage3Recommender

recommender = Stage3Recommender().fit(recipes, train)

profile = UserProfile(diet="vegan", allergens=["nuts", "dairy", "gluten"])
context = SessionContext(pantry=["tomato", "pasta", "garlic"], max_minutes=30, meal_intent="main")
known_user = int(train["user_id"].iloc[0])

recs = recommender.recommend(profile, context, user_id=known_user, top_n=5)
for rec in recs:
    print(f"{rec.final_score:.3f} | {rec.name}")
    print(f"  why: {rec.explanation}")

In [ ]:
# 6) Ablation eval (increase sample for final report)
from src.eval.offline_eval import run_ablation
import pandas as pd

validation = load_interaction_split("validation")

USER_SAMPLE = 100   # try 500 for report
TOP_K = 10
# recipes already loaded (5k sample). For full catalog:
# recipes = load_recipes(columns=["id", "name", "ingredients", "minutes", "tags"])

results = run_ablation(
    recipes=recipes,
    train_interactions=train,
    held_out=validation,
    user_sample_size=USER_SAMPLE,
    k=TOP_K,
)

display(results.sort_values(by=f"hit_rate@{TOP_K}", ascending=False))

## Full catalog run

Uncomment below only if you accept long runtimes:

```python
recipes_full = load_recipes(columns=["id", "name", "ingredients", "minutes", "tags"])
recommender = Stage3Recommender().fit(recipes_full, train)
results_full = run_ablation(recipes=recipes_full, train_interactions=train, held_out=validation, user_sample_size=500, k=10)
results_full.to_csv("ablation_full.csv", index=False)
```